# RQ5: Charging Efficiency by Location and Charger Status

This Kaggle notebook compares charging efficiency, time, cost, and risk by location and charger working status.

**Outputs saved to `/kaggle/working/rq5_location_charger_status/`:**
- `RQ5_charging_location_status_table.csv`
- `RQ5_charging_efficiency_by_location_status_figure.pdf`

**Research question:** How do charger working status and charging location relate to charging efficiency, charging time, charging cost, and risk rate?

## Run this cell to generate the actual answer, CSV table, and PDF figure

The notebook automatically searches for the dataset file inside `/kaggle/input`. If it cannot find it, set `DATASET_PATH` manually in the code cell.

In [ ]:

# ============================================================
# Common setup: imports, paths, loading, cleaning, preprocessing
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

RANDOM_STATE = 42

# Change this manually only if Kaggle cannot auto-detect your file.
# Example:
# DATASET_PATH = "/kaggle/input/ev-charging-behavior-analysis-and-financial-risk/your_file.csv"
DATASET_PATH = None

def file_has_expected_columns(path):
    """
    Checks whether a candidate file looks like the raw EV dataset rather than
    an output table generated by one of the notebooks.
    """
    try:
        if path.lower().endswith(".csv"):
            cols = pd.read_csv(path, nrows=0).columns.astype(str).str.strip().tolist()
        elif path.lower().endswith((".xlsx", ".xls")):
            cols = pd.read_excel(path, nrows=0).columns.astype(str).str.strip().tolist()
        else:
            return False
    except Exception:
        return False

    required = {"High_Default_Risk", "Charging_Efficiency_Index"}
    return required.issubset(set(cols)) and len(cols) >= 10

def find_dataset_file():
    """
    Finds the raw CSV or Excel dataset file automatically.
    Works in Kaggle by searching /kaggle/input first.
    Also avoids accidentally selecting CSV output tables generated by the notebooks.
    """
    if DATASET_PATH is not None and os.path.exists(DATASET_PATH):
        return DATASET_PATH

    roots = ["/kaggle/input", ".", "/mnt/data"]
    extensions = (".csv", ".xlsx", ".xls")
    candidates = []

    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            # Skip common output folders when running locally
            if any(part.startswith("rq") and "output" in part for part in dirpath.lower().split(os.sep)):
                continue
            for filename in filenames:
                lower = filename.lower()
                if lower.endswith(extensions):
                    candidates.append(os.path.join(dirpath, filename))

    if not candidates:
        raise FileNotFoundError(
            "No CSV/XLS/XLSX file found. Add the Kaggle dataset to the notebook "
            "or set DATASET_PATH manually."
        )

    # First priority: files that actually contain the required raw dataset columns.
    valid_raw_files = [path for path in candidates if file_has_expected_columns(path)]
    if valid_raw_files:
        keywords = ["ev", "charging", "financial", "risk", "behavior"]
        preferred = [
            path for path in valid_raw_files
            if any(word in os.path.basename(path).lower() for word in keywords)
        ]
        return sorted(preferred or valid_raw_files)[0]

    # Fallback: use filename keywords if header check cannot be completed.
    keywords = ["ev", "charging", "financial", "risk", "behavior"]
    preferred = [
        path for path in candidates
        if any(word in os.path.basename(path).lower() for word in keywords)
    ]

    selected = sorted(preferred or candidates)[0]
    return selected

def load_raw_dataset():
    path = find_dataset_file()
    print(f"Loading dataset from: {path}")

    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    elif path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    else:
        raise ValueError("Unsupported file type. Use CSV, XLSX, or XLS.")

    print(f"Raw dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    return df

def clean_dataset(df):
    """
    Basic cleaning for this EV charging dataset:
    - strips column names and string values
    - removes duplicate rows
    - converts known numeric columns
    - converts impossible negative values into missing values
    - standardizes binary columns where needed
    """
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    df = df.drop_duplicates()

    # Strip text columns
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

    # Standardize binary columns if they came in as text before numeric conversion
    binary_maps = {
        "yes": 1, "y": 1, "true": 1, "high": 1, "risk": 1, "default": 1, "1": 1,
        "no": 0, "n": 0, "false": 0, "low": 0, "non-risk": 0, "non risk": 0, "0": 0
    }
    for col in ["Loan_Taken", "High_Default_Risk"]:
        if col in df.columns and df[col].dtype == "object":
            df[col] = df[col].astype(str).str.lower().map(binary_maps)

    known_numeric = [
        "User_ID", "Age", "Battery_Capacity_kWh", "Charging_Sessions_Per_Month",
        "Avg_Charge_Cost", "Distance_Travelled_Per_Month", "Loan_Taken",
        "Missed_Payments_Last_6M", "Tenure_Months", "App_Usage_Score",
        "Charging_Time_Minutes", "High_Default_Risk", "Charging_Efficiency_Index"
    ]
    for col in known_numeric:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Treat impossible negative values as missing
    non_negative_cols = [
        "Age", "Battery_Capacity_kWh", "Charging_Sessions_Per_Month",
        "Avg_Charge_Cost", "Distance_Travelled_Per_Month",
        "Missed_Payments_Last_6M", "Tenure_Months", "App_Usage_Score",
        "Charging_Time_Minutes", "Charging_Efficiency_Index"
    ]
    for col in non_negative_cols:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    # Keep app score in a practical range if the column exists
    if "App_Usage_Score" in df.columns:
        df.loc[(df["App_Usage_Score"] < 0) | (df["App_Usage_Score"] > 10), "App_Usage_Score"] = np.nan

    return df

def make_output_dir(name):
    output_dir = os.path.join("/kaggle/working" if os.path.exists("/kaggle/working") else ".", name)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output folder: {output_dir}")
    return output_dir

def split_features_target(df, target_col, drop_cols=None):
    if drop_cols is None:
        drop_cols = []
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available columns: {list(df.columns)}")

    data = df.dropna(subset=[target_col]).copy()

    # Force target to integer binary if classification
    if target_col == "High_Default_Risk":
        data = data[data[target_col].isin([0, 1])].copy()
        data[target_col] = data[target_col].astype(int)

    y = data[target_col]
    X = data.drop(columns=[target_col] + [c for c in drop_cols if c in data.columns])
    return X, y, data

def get_feature_types(X):
    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]
    return numeric_cols, categorical_cols

def make_preprocessor(X, scale_numeric=True):
    numeric_cols, categorical_cols = get_feature_types(X)

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)

    try:
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
    except TypeError:
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
        ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols)
        ],
        remainder="drop"
    )
    return preprocessor, numeric_cols, categorical_cols

def get_transformed_feature_names(preprocessor, numeric_cols, categorical_cols):
    feature_names = list(numeric_cols)
    if categorical_cols:
        try:
            ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
            cat_names = list(ohe.get_feature_names_out(categorical_cols))
        except Exception:
            cat_names = []
            for col in categorical_cols:
                cat_names.append(col)
        feature_names.extend(cat_names)
    return feature_names

def save_table(df, output_dir, filename):
    path = os.path.join(output_dir, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path

def save_figure(output_dir, filename):
    path = os.path.join(output_dir, filename)
    plt.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path

def style_plot(title, subtitle=None, xlabel=None, ylabel=None):
    plt.title(title, fontsize=15, fontweight="bold", pad=18)
    if subtitle:
        plt.suptitle(subtitle, fontsize=10, y=0.94, color="#3b4f6b")
    if xlabel:
        plt.xlabel(xlabel, fontsize=11)
    if ylabel:
        plt.ylabel(ylabel, fontsize=11)
    plt.grid(axis="y", linestyle="--", alpha=0.35)
    plt.gca().set_axisbelow(True)

def print_dataset_summary(df):
    print("Cleaned dataset shape:", df.shape)
    print("\nColumns:")
    print(list(df.columns))
    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(20))


# ============================================================
# RQ5: Charging efficiency by location and charger status
# ============================================================

output_dir = make_output_dir("rq5_location_charger_status")

df_raw = load_raw_dataset()
df = clean_dataset(df_raw)
print_dataset_summary(df)

required_cols = [
    "Charging_Location_Type", "Charger_Working_Status", "Charging_Efficiency_Index",
    "Charging_Time_Minutes", "Avg_Charge_Cost", "High_Default_Risk"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns required for RQ5: {missing}")

analysis_df = df[required_cols].dropna(subset=["Charging_Location_Type", "Charger_Working_Status"]).copy()
analysis_df = analysis_df[analysis_df["High_Default_Risk"].isin([0, 1])].copy()

summary = (
    analysis_df
    .groupby(["Charging_Location_Type", "Charger_Working_Status"], dropna=False)
    .agg(
        N_Users=("High_Default_Risk", "size"),
        Mean_Efficiency_Index=("Charging_Efficiency_Index", "mean"),
        Mean_Charging_Time_Minutes=("Charging_Time_Minutes", "mean"),
        Mean_Charge_Cost=("Avg_Charge_Cost", "mean"),
        High_Default_Risk_Percent=("High_Default_Risk", lambda x: 100 * np.mean(x))
    )
    .reset_index()
)

for col in ["Mean_Efficiency_Index", "Mean_Charging_Time_Minutes", "Mean_Charge_Cost", "High_Default_Risk_Percent"]:
    summary[col] = summary[col].round(2)

summary = summary.sort_values(["Charging_Location_Type", "Charger_Working_Status"]).reset_index(drop=True)

display(summary)
save_table(summary, output_dir, "RQ5_charging_location_status_table.csv")

best_eff = summary.sort_values("Mean_Efficiency_Index", ascending=False).iloc[0]
worst_eff = summary.sort_values("Mean_Efficiency_Index", ascending=True).iloc[0]
print(
    f"Actual answer for RQ5: The highest mean efficiency is observed for "
    f"{best_eff['Charging_Location_Type']} / {best_eff['Charger_Working_Status']} "
    f"({best_eff['Mean_Efficiency_Index']}). The lowest mean efficiency is observed for "
    f"{worst_eff['Charging_Location_Type']} / {worst_eff['Charger_Working_Status']} "
    f"({worst_eff['Mean_Efficiency_Index']})."
)

# Figure 5: Grouped bar chart
pivot = summary.pivot(
    index="Charging_Location_Type",
    columns="Charger_Working_Status",
    values="Mean_Efficiency_Index"
).fillna(0)

# Put common order if present
location_order = [x for x in ["Private", "Public", "Highway"] if x in pivot.index] + [x for x in pivot.index if x not in ["Private", "Public", "Highway"]]
pivot = pivot.loc[location_order]

status_order = [x for x in ["Working", "Not Working"] if x in pivot.columns] + [x for x in pivot.columns if x not in ["Working", "Not Working"]]
pivot = pivot[status_order]

x = np.arange(len(pivot.index))
width = 0.8 / max(1, len(pivot.columns))

plt.figure(figsize=(10, 6))
colors = ["#2f72c4", "#8aa6bf", "#0b2f63", "#bcc7d3"]

for i, status in enumerate(pivot.columns):
    offsets = x - 0.4 + width/2 + i * width
    bars = plt.bar(offsets, pivot[status].values, width=width, label=status, color=colors[i % len(colors)])
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height + 1, f"{height:.1f}", ha="center", va="bottom", fontsize=9)

plt.xticks(x, pivot.index)
plt.ylabel("Mean charging efficiency index")
plt.xlabel("Charging location")
plt.ylim(0, max(100, pivot.max().max() * 1.15))
plt.title("Figure 5. Charging efficiency by location and charger status", fontsize=15, fontweight="bold", pad=18)
plt.grid(axis="y", linestyle="--", alpha=0.35)
plt.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=len(pivot.columns), frameon=False)
plt.figtext(
    0.5, -0.04,
    "Note: Bars show mean charging efficiency by charging location and charger working status.",
    ha="center", fontsize=9
)
plt.tight_layout()

save_figure(output_dir, "RQ5_charging_efficiency_by_location_status_figure.pdf")
plt.show()
